## 03_graph_analytics.ipynb

### Graph Analytics using Neo4j GDS

This notebook performs graph analytics on the supply-chain graph
using Neo4j Graph Data Science (GDS).

Algorithms:
- PageRank
- Louvain Community Detection

In [1]:
import pandas as pd
from neo4j import GraphDatabase

In [2]:
URI = "neo4j://127.0.0.1:7687" 
AUTH = ("neo4j", "Swara@1212")
driver = GraphDatabase.driver( URI, auth=AUTH )

In [3]:
with driver.session() as session:
    result = session.run("RETURN 'Neo4j Connected' AS message")
    print(result.single()["message"])

Neo4j Connected


In [4]:
projection_query = """
CALL gds.graph.project(
    'supplyGraph',

    ['Orders', 'Customers', 'Products'],

    {
        PLACED: {
            orientation: 'UNDIRECTED'
        },

        CONTAINS: {
            orientation: 'UNDIRECTED'
        }
    }
)
"""

In [5]:
# -----------------------------------
# Check Existing GDS Graphs
# -----------------------------------

check_graph_query = """
CALL gds.graph.list()
YIELD graphName
RETURN graphName
"""

with driver.session() as session:

    result = session.run(check_graph_query)

    graphs_df = pd.DataFrame(
        [dict(record) for record in result]
    )

graphs_df

""


In [6]:
# -----------------------------------
# Drop Existing Projection
# -----------------------------------

drop_query = """
CALL gds.graph.drop('supplyGraph', false)
"""

with driver.session() as session:

    try:
        result = session.run(drop_query)

        for record in result:
            print(record)

        print("Old graph projection dropped.")

    except Exception as e:
        print("No existing graph found or already dropped.")
        print(e)

# -----------------------------------
# Recreate Graph Projection
# -----------------------------------

projection_query = """
CALL gds.graph.project(

    'supplyGraph',

    ['Order', 'Product', 'Category', 'Department', 'Region', 'ShippingMode'],

    {
        CONTAINS: {
            orientation: 'UNDIRECTED'
        },

        SHIPPED_TO: {
            orientation: 'UNDIRECTED'
        },

        USED_MODE: {
            orientation: 'UNDIRECTED'
        },

        IN_CATEGORY: {
            orientation: 'UNDIRECTED'
        },

        IN_DEPARTMENT: {
            orientation: 'UNDIRECTED'
        }
    }
)
"""

with driver.session() as session:

    result = session.run(projection_query)

    for record in result:
        print(record)

print("New graph projection created successfully.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL gds.graph.drop('supplyGraph', false)\n"


Old graph projection dropped.
<Record nodeProjection={'Order': {'properties': {}, 'label': 'Order'}, 'Product': {'properties': {}, 'label': 'Product'}, 'Region': {'properties': {}, 'label': 'Region'}, 'ShippingMode': {'properties': {}, 'label': 'ShippingMode'}, 'Department': {'properties': {}, 'label': 'Department'}, 'Category': {'properties': {}, 'label': 'Category'}} relationshipProjection={'SHIPPED_TO': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'SHIPPED_TO', 'properties': {}, 'indexInverse': False}, 'CONTAINS': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'CONTAINS', 'properties': {}, 'indexInverse': False}, 'IN_CATEGORY': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'IN_CATEGORY', 'properties': {}, 'indexInverse': False}, 'USED_MODE': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'USED_MODE', 'properties': {}, 'indexInverse': False}, 'IN_DEPARTMENT': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT',

In [7]:
# -----------------------------------
# Run PageRank
# -----------------------------------

pagerank_query = """
CALL gds.pageRank.write(
    'supplyGraph',
    {
        writeProperty: 'pagerank'
    }
)
YIELD nodePropertiesWritten, ranIterations

RETURN
    nodePropertiesWritten,
    ranIterations
"""

with driver.session() as session:

    result = session.run(pagerank_query)

    pagerank_stats = pd.DataFrame(
        [dict(record) for record in result]
    )

pagerank_stats

,nodePropertiesWritten,ranIterations
0,65958,20


In [8]:
# -----------------------------------
# Top 10 Orders by PageRank
# -----------------------------------

top_pagerank_query = """
MATCH (o:Order)

WHERE o.pagerank IS NOT NULL

RETURN
    o.order_id AS node_id,
    o.pagerank AS pagerank

ORDER BY pagerank DESC

LIMIT 10
"""

with driver.session() as session:

    result = session.run(top_pagerank_query)

    pagerank_df = pd.DataFrame(
        [dict(record) for record in result]
    )

pagerank_df

,node_id,pagerank
0,68703,0.745220
1,68733,0.738587
2,68773,0.738089
3,68710,0.737655
4,68780,0.737393
5,68786,0.737063
6,68227,0.734355
7,68316,0.734049
8,68453,0.732718
9,68858,0.732707


In [9]:
# -----------------------------------
# Louvain Community Detection
# -----------------------------------

community_query = """
CALL gds.louvain.write(
    'supplyGraph',
    {
        writeProperty: 'community'
    }
)
YIELD communityCount, modularity

RETURN
    communityCount,
    modularity
"""

with driver.session() as session:

    result = session.run(community_query)

    community_stats_df = pd.DataFrame(
        [dict(record) for record in result]
    )

community_stats_df

,communityCount,modularity
0,30,0.233182


In [10]:
# -----------------------------------
# Top Community Sizes
# -----------------------------------

community_size_query = """
MATCH (n)

WHERE n.community IS NOT NULL

RETURN
    n.community AS community,
    count(*) AS size

ORDER BY size DESC

LIMIT 10
"""

with driver.session() as session:

    result = session.run(community_size_query)

    community_df = pd.DataFrame(
        [dict(record) for record in result]
    )

community_df

,community,size
0,182,13866
1,178,12886
2,168,12298
3,87,3454
4,184,3160
5,179,2331
6,171,1805
7,50,1547
8,43,1399
9,106,1153


In [11]:
# -----------------------------------
# Modularity Score
# -----------------------------------

modularity_query = """
CALL gds.louvain.stats('supplyGraph')

YIELD modularity

RETURN modularity
"""

with driver.session() as session:

    result = session.run(modularity_query)

    modularity_df = pd.DataFrame(
        [dict(record) for record in result]
    )

modularity_df

,modularity
0,0.233182


In [12]:
# -----------------------------------
# Export Graph Features
# -----------------------------------

gds_features_query = """
MATCH (o:Order)

WHERE o.pagerank IS NOT NULL
AND o.community IS NOT NULL

RETURN
    o.order_id AS node_id,
    o.pagerank AS pagerank,
    o.community AS community
"""

with driver.session() as session:

    result = session.run(gds_features_query)

    gds_features_df = pd.DataFrame(
        [dict(record) for record in result]
    )

gds_features_df.head()

,node_id,pagerank,community
0,41777,0.721510,50
1,27559,0.408641,168
2,47394,0.399931,41
3,30336,0.487441,168
4,20602,0.568930,91


In [13]:
# -----------------------------------
# Create Clean Data Directory
# -----------------------------------

import os

os.makedirs(
    "../data/clean",
    exist_ok=True
)

print("Directory created successfully.")

Directory created successfully.


In [14]:
# -----------------------------------
# Save Features CSV
# -----------------------------------

gds_features_df.to_csv(
    "../data/clean/gds_features.csv",
    index=False
)

print("Graph features exported successfully.")

Graph features exported successfully.


In [15]:
import pandas as pd

check_df = pd.read_csv(
    "../data/clean/gds_features.csv"
)

check_df.head()

,node_id,pagerank,community
0,41777,0.721510,50
1,27559,0.408641,168
2,47394,0.399931,41
3,30336,0.487441,168
4,20602,0.568930,91


In [16]:
# -----------------------------------
# Drop Graph Projection
# -----------------------------------

drop_query = """
CALL gds.graph.drop('supplyGraph')
"""

with driver.session() as session:

    result = session.run(drop_query)

    for record in result:
        print(record)

print("Graph projection dropped.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL gds.graph.drop('supplyGraph')\n"


<Record graphName='supplyGraph' database='neo4j' databaseLocation='local' memoryUsage='' sizeInBytes=-1 nodeCount=65958 relationshipCount=583006 configuration={'relationshipProjection': {'SHIPPED_TO': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'SHIPPED_TO', 'properties': {}, 'indexInverse': False}, 'CONTAINS': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'CONTAINS', 'properties': {}, 'indexInverse': False}, 'IN_CATEGORY': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'IN_CATEGORY', 'properties': {}, 'indexInverse': False}, 'USED_MODE': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'USED_MODE', 'properties': {}, 'indexInverse': False}, 'IN_DEPARTMENT': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'IN_DEPARTMENT', 'properties': {}, 'indexInverse': False}}, 'jobId': 'jid-12974943-6a00-4c9a-9b7b-2c62a1dbf474', 'nodeProjection': {'Order': {'properties': {}, 'label': 'Order'}, 'Product': {'properti

In [17]:
# -----------------------------------
# Close Neo4j Driver
# -----------------------------------

driver.close()

print("Neo4j connection closed.")

Neo4j connection closed.


### Business Insights
#### PageRank Analysis

PageRank identified highly influential nodes in the supply-chain network. Orders and products with high PageRank scores indicate operational importance and strong connectivity within the ecosystem.

#### Community Detection
Louvain community detection revealed clusters of related supply-chain entities. These communities may represent:
related product groups
shipment patterns
regional logistics clusters
operational dependencies
These insights can help optimize supply-chain operations and improve strategic decision-making.